<div style="border-left:4px solid #fbbf24;padding:2px 0 2px 16px;margin:6px 0 18px;"><div style="font:800 27px/1.15 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;letter-spacing:-0.02em;">NL2SQL <span style="font-weight:500;color:#fbbf24;">Run All</span></div><div style="font:400 15px/1.55 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#71717a;margin-top:5px;">The whole pipeline end to end, then the live service.</div></div>

[Setup](https://www.kaggle.com/code/kirazul/nl2sql-1-setup) &nbsp;|&nbsp; [Understanding](https://www.kaggle.com/code/kirazul/nl2sql-2-understanding) &nbsp;|&nbsp; [Architectures](https://www.kaggle.com/code/kirazul/nl2sql-3-architectures) &nbsp;|&nbsp; **Run All**

## 1. Setup

The code is cloned from GitHub. The database, the index and the two models are
read from [notebook 1](https://www.kaggle.com/code/kirazul/nl2sql-1-setup)'s saved output, where they already are. Nothing
is downloaded or rebuilt here.

Before running: **Add Input > Notebook Output > NL2SQL 1 Setup**, add the secrets
listed below under **Add-ons > Secrets**, and enable Internet.

In [ ]:
%%capture --no-stderr
!pip install -q --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu \
    "llama-cpp-python>=0.3" "gliner2>=1.3" "langgraph>=1.0" "langsmith>=0.10" \
    "fastapi>=0.115" "uvicorn[standard]>=0.34" "pydantic-settings>=2.6" \
    "sqlglot>=25.0" "rapidfuzz>=3.10" "pyyaml>=6.0" "httpx>=0.27" "python-dotenv>=1.0"

In [ ]:
import os, re, sys, json, time, shutil, subprocess
from pathlib import Path

ON_KAGGLE = Path("/kaggle").exists()
WORK      = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
INPUTS    = Path("/kaggle/input")
REPO      = "https://github.com/Kirazul/NL2SQL-demo.git"

SECRETS = {
    "GITHUB_TOKEN":       "clone the code (the repository is private)",
    "GROQ_API_KEY":       "the cloud model that writes the SQL",
    "OPENROUTER_API_KEY": "fallback when Groq rate-limits",
    "LANGSMITH_API_KEY":  "tracing backend",
    "PUBLISH_TOKEN":      "announce this session to the web interface",
}
REQUIRED = ('GROQ_API_KEY',)


WHY = {}          # label -> why it could not be read, when it could not


def secret(label, default=""):
    """One secret, by label. Kaggle grants access per notebook, not per account.

    The reason a lookup failed is kept rather than swallowed: "not attached to
    this notebook" and "the backend refused" both end as an empty string, and
    without the reason the two are indistinguishable from the output.
    """
    if ON_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            value = UserSecretsClient().get_secret(label)
            if value:
                return value
            WHY[label] = "Kaggle returned an empty value"
        except Exception as error:
            WHY[label] = f"{type(error).__name__}: {str(error)[:110]}"
    return os.environ.get(label, default)


def load_secrets(project=None):
    """Read every label into the environment and print what was found.

    An empty secret is removed rather than set blank, so the package falls back to
    its own default instead of an empty string.
    """
    local = {}
    if not ON_KAGGLE and project and (project / ".env").exists():
        for line in (project / ".env").read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                label, _, value = line.partition("=")
                local[label.strip()] = value.strip().strip("\"'")

    for label in SECRETS:
        value = secret(label) or local.get(label, "")
        if value:
            os.environ[label] = value
        else:
            os.environ.pop(label, None)

    for label, purpose in SECRETS.items():
        if os.environ.get(label):
            state = "ok"
        elif label in REQUIRED:
            state = "REQUIRED"
        else:
            state = "-"
        print(f"  {label:<20}{state:<10}{purpose}")

    if WHY:
        print("\n  why a secret could not be read")
        for label, reason in WHY.items():
            print(f"    {label:<20}{reason}")

    absent = [l for l in REQUIRED if not os.environ.get(l)]
    if absent:
        where = "Add-ons > Secrets, in this notebook" if ON_KAGGLE else ".env"
        print(f"\n  Missing: {', '.join(absent)}. Set it in {where} and run this cell again.")
    return not absent


def get_code():
    """Clone the repository into a writable directory and put it on the path.

    Kaggle mounts every input read-only and notebook 1 writes a database next to
    the package, so the code never runs from where it is mounted.
    """
    if (Path.cwd() / "src/hybridsql").exists():
        return Path.cwd()

    target = WORK / "nl2sql"
    if (target / "src/hybridsql").exists():
        return target

    token = secret("GITHUB_TOKEN")
    url = REPO.replace("https://", f"https://{token}@") if token else REPO
    done = subprocess.run(["git", "clone", "--depth", "1", "--quiet", url, str(target)],
                          capture_output=True, text=True)
    if done.returncode:
        detail = done.stderr.replace(token, "***") if token else done.stderr
        raise SystemExit(
            "Could not clone the repository.\n\n"
            "  It is private, so this notebook needs a GITHUB_TOKEN secret:\n"
            "  Add-ons > Secrets > attach GITHUB_TOKEN, then run this cell again.\n\n"
            "  Kaggle grants a secret one notebook at a time. Attaching it in\n"
            "  another notebook does not attach it here.\n\n" + detail
        )
    return target
ARTEFACTS = {
    "database":    ("data/warehouse/eicu.db",                   None),
    "value index": ("data/warehouse/value_index.db",            None),
    "GLiNER2":     ("models/gliner2-base-v1",                   "model.safetensors"),
    "Qwen3-1.7B":  ("models/qwen3-1.7b/Qwen3-1.7B-Q4_K_M.gguf", None),
}
DEPTHS = ("", "*/", "*/*/", "*/*/*/", "*/*/*/*/", "*/*/*/*/*/")


def whole(path, probe=None):
    """Present and finished. A model directory with no weights in it is neither."""
    return (path / probe).exists() if probe else path.exists()


def find_input(relative, probe=None):
    """The first attached input carrying `relative`, at whatever depth it sits."""
    if not INPUTS.exists():
        return None
    for prefix in DEPTHS:
        for hit in sorted(INPUTS.glob(prefix + relative)):
            if whole(hit, probe):
                return hit
    return None


def attached_inputs():
    """The inputs actually attached, named by what they carry rather than by the
    directory level Kaggle happens to mount them under."""
    if not INPUTS.exists():
        return []
    markers = ("src", "data", "models", "nl2sql")
    return [p.relative_to(INPUTS).as_posix()
            for pattern in ("*", "*/*", "*/*/*")
            for p in sorted(INPUTS.glob(pattern))
            if p.is_dir() and any((p / m).exists() for m in markers)]


def locate(project):
    """Every artefact, in the working copy or in an attached input."""
    found, missing = {}, []
    for label, (relative, probe) in ARTEFACTS.items():
        if label == "value index":
            continue                       # always beside the database, see below
        local = project / relative
        path = local if whole(local, probe) else find_input(relative, probe)
        (found.__setitem__(label, path) if path else missing.append(label))

    # The package derives the index path from the database path, so the two must
    # be in the same directory. Looking for it anywhere else would resolve here
    # and fail there.
    if "database" in found:
        index = found["database"].with_name("value_index.db")
        found["value index"] = index if index.exists() else missing.append("value index")
    else:
        missing.append("value index")
    return found, [m for m in missing if m]


def configure(found):
    os.environ["DB_PATH"]             = str(found["database"])
    os.environ["GLINER_MODEL"]        = str(found["GLiNER2"])
    os.environ["LOCAL_LLM_GGUF_PATH"] = str(found["Qwen3-1.7B"])
    os.environ["LOCAL_LLM_THREADS"]   = str(max(2, os.cpu_count() or 4))
    os.environ["LOCAL_LLM_BACKEND"]   = "llamacpp"
    os.environ["PRIVACY_MODE"]        = "demo"
    os.environ["LANGSMITH_PROJECT"]   = "nl2sql"
    os.environ["LANGSMITH_TRACING"]   = "1" if os.environ.get("LANGSMITH_API_KEY") else "0"


def size_mb(path):
    if path.is_dir():
        return sum(f.stat().st_size for f in path.rglob("*") if f.is_file()) / 1e6
    return path.stat().st_size / 1e6 if path.exists() else 0.0


def show(found):
    for label, (relative, _) in ARTEFACTS.items():
        path = found.get(label)
        if path is None:
            print(f"  {label:<14}{'missing':>10}")
            continue
        root = path.parents[len(Path(relative).parts) - 1]
        if WORK in path.parents:
            where = "built here"
        elif INPUTS.exists() and (INPUTS in root.parents or root == INPUTS):
            where = root.relative_to(INPUTS).as_posix()
        else:
            where = str(root)
        print(f"  {label:<14}{size_mb(path):>9.0f} MB   {where}")
print("code")
PROJECT = get_code()
sys.path.insert(0, str(PROJECT / "src"))
os.chdir(PROJECT)
print(f"  {PROJECT}")

print("\nsecrets")
load_secrets(PROJECT)

FOUND, MISSING = locate(PROJECT)
if MISSING:
    raise SystemExit(
        "Notebook 1's output is not attached, and nothing is built in this notebook.\n"
        f"  missing:  {', '.join(MISSING)}\n"
        f"  attached: {attached_inputs() or 'nothing'}\n\n"
        "  Add Input > Notebook Output > NL2SQL 1 Setup\n"
        "  https://www.kaggle.com/code/kirazul/nl2sql-1-setup"
    )

configure(FOUND)
print("\nartefacts")
show(FOUND)

---

## 2. The cloud provider

In [ ]:
from hybridsql.providers import cloud

targets = cloud.chain()
for target in targets:
    print(f"  {target.name}")

if not targets:
    raise SystemExit(
        "No cloud provider is configured. Three of the four architectures call one,\n"
        "and without a key they fail instantly with no tokens and no rows.\n"
        "Set GROQ_API_KEY under Add-ons > Secrets, attached to this notebook,\n"
        "then run the setup cell again."
    )
print(f"\n  {len(targets)} target(s), tried in this order")

---

## 3. The four architectures

Three questions through each of the four. [Notebook 2](https://www.kaggle.com/code/kirazul/nl2sql-2-understanding) takes the
understanding stage apart, [notebook 3](https://www.kaggle.com/code/kirazul/nl2sql-3-architectures) the four designs; this one just
runs them. Free Groq allows 30 requests a minute, so the loop paces itself.

In [ ]:
from hybridsql.graph import ARMS, run
from hybridsql.graph.state import public

QUESTIONS = [
    "How many patients received aspirin?",
    "What is the average age of patients admitted to the MICU?",
    "How many female patients were discharged alive?",
]

results = []


def ask(question, arm, write=False):
    """Run one question through one architecture and print every stage of it."""
    r = public(run(question, arm=arm, write=write))
    results.append(r)

    print(f"  question    {question}")
    if r["masked_question"]:
        print(f"  sent        {r['masked_question']}")
    if r["opaque"].get("question"):
        print(f"  relabelled  {r['opaque']['question']}")
    if r["sql"]:
        print(f"  sql         {' '.join(r['sql'].split())}")
    if r["success"]:
        print(f"  rows        {r['row_count']}")
    else:
        print(f"  failed      {r['failed_stage']}: {r['failure_reason']}")
    if r["answer"]:
        print(f"  answer      {' '.join(r['answer'].split())[:180]}")
    print(f"  cost        {sum(r['ms'].values()):.0f} ms, {r['cloud_tokens']} cloud tokens, "
          f"{r['egress_chars']} chars sent, {r['egress_values']} values sent\n")
    return r

In [ ]:
for i, question in enumerate(QUESTIONS, 1):
    print(f"[{i}/{len(QUESTIONS)}] {question}\n")
    for arm in ARMS:
        ask(question, arm)
    time.sleep(2.5)

**What Hybrid Opaque actually sent.** Labels only, redrawn for this request.

In [ ]:
opaque = next((r["opaque"] for r in results
               if r["arm"] == "hybrid_opaque" and r["opaque"].get("question")), None)
if opaque:
    print(f"  relabelled  {opaque['question']}")
    print(f"  parameters  {opaque['parameters'].strip()}")
    print(f"  schema      {opaque['tables']} tables, {opaque['columns']} columns, all labels")
    print("\n  what the labels meant, kept here:")
    for alias, real in opaque["labels"].items():
        print(f"    {alias:<6}{real}")

### The answer, written here

The weights are already loaded from the Full Local runs, so this is the writing
step on its own: rows turned into a sentence without leaving the machine.

In [ ]:
written = public(run(QUESTIONS[0], arm="hybrid", write=True))
print(f"  sql written by     {written['sql_author']}")
print(f"  answer written by  {written['answer_author']}")
print(f"\n  {written['answer']}")

---

## 5. The comparison

`ran` counts queries that executed. The column that decides is **values sent**.

In [ ]:
from collections import defaultdict

rows = defaultdict(lambda: {"n": 0, "ok": 0, "ms": 0.0, "values": 0, "chars": 0, "tokens": 0})
for r in results:
    e = rows[r["arm"]]
    e["n"] += 1
    e["ok"] += int(r["success"])
    e["ms"] += sum(r["ms"].values())
    e["values"] += r["egress_values"]
    e["chars"] += r["egress_chars"]
    e["tokens"] += r["cloud_tokens"]

head = f"{'architecture':<16}{'ran':>7}{'avg ms':>9}{'values sent':>13}{'chars sent':>12}{'tokens':>8}"
print(head)
print("-" * len(head))
for arm in ARMS:
    e = rows[arm]
    if e["n"]:
        print(f"{arm:<16}{str(e['ok']) + '/' + str(e['n']):>7}{e['ms'] / e['n']:>9.0f}"
              f"{e['values']:>13}{e['chars']:>12}{e['tokens']:>8}")

failed = [r for r in results if not r["success"]]
if failed:
    print("\nwhat failed")
    for r in failed:
        print(f"  {r['arm']:<16}{r['failed_stage']}: {r['failure_reason'][:70]}")

---

## 5. The live service

The same pipeline, behind an address a browser can reach.

The interface is served by a Cloudflare Worker at **https://nl2sql.eclipse-kira.workers.dev**. A Kaggle session
has no stable address: `cloudflared` gives this notebook a public hostname, but a
new one every restart. So the notebook publishes its current address to the
Worker, and the page reads it back. That is the only thing the Worker does.

**It does not carry the conversation.** Your browser connects straight to this
session, and the question, the SQL and the answer never pass through Cloudflare.
Proxying would have been easier, one origin and no CORS, but this project's claim
is that the data stays inside a known boundary, and routing every answer through
a third party to tidy up a URL would have contradicted it.

| Endpoint, served here | Purpose |
|---|---|
| `GET /health` | component status |
| `GET /meta` | schema summary, available architectures |
| `POST /ask` | one question, one answer |
| `POST /ask/stream` | one event per stage |
| `POST /compare` | the same question through all four |
| `GET /egress/report` | the audit journal |

**Start the API and the tunnel.**

In [ ]:
import threading, uvicorn, httpx
from hybridsql.api.app import app

WORKER = "https://nl2sql.eclipse-kira.workers.dev"

threading.Thread(
    target=lambda: uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning"),
    daemon=True,
).start()

for _ in range(60):
    try:
        if httpx.get("http://127.0.0.1:8000/health", timeout=2).status_code == 200:
            print("  api       listening on 8000")
            break
    except Exception:
        time.sleep(0.5)
else:
    raise SystemExit("the API did not start")

if not Path("cloudflared").exists():
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared

log = Path("tunnel.log")
log.unlink(missing_ok=True)
tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=log.open("w"), stderr=subprocess.STDOUT,
)

public_url = ""
for _ in range(60):
    time.sleep(1)
    if log.exists():
        match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log.read_text())
        if match:
            public_url = match.group(0)
            break
assert public_url, "the tunnel did not start, see tunnel.log"
print(f"  tunnel    {public_url}")

**Publish the address, then check it.** Three things have to hold, and each fails
differently, so each is checked and named.

In [ ]:
ok = True
token = os.environ.get("PUBLISH_TOKEN", "")

if not token:
    ok = False
    print("  announce   skipped, PUBLISH_TOKEN is not set")
    print("             It must be the same value in two places: a Kaggle secret")
    print("             on this notebook, and `wrangler secret put PUBLISH_TOKEN`")
    print("             in deploy/worker.")
else:
    r = httpx.post(f"{WORKER}/api/backend",
                   headers={"Authorization": f"Bearer {token}"},
                   json={"url": public_url, "label": "kaggle run-all"}, timeout=20)
    if r.status_code == 200:
        print(f"  announce   ok, {r.json().get('url')}")
    elif r.status_code == 401:
        ok = False
        print("  announce   refused (401): this notebook's PUBLISH_TOKEN and the")
        print("             Worker's are different values")
    else:
        ok = False
        print(f"  announce   failed, HTTP {r.status_code} {r.text.strip()[:100]}")

try:
    entry = httpx.get(f"{WORKER}/api/backend", timeout=20).json()
except Exception as error:
    ok, entry = False, {}
    print(f"  rendezvous unreachable, {type(error).__name__}")

if entry.get("online") and entry.get("url", "").rstrip("/") == public_url.rstrip("/"):
    print(f"  rendezvous ok, the page will be sent to {entry['url']}")
elif entry.get("online"):
    ok = False
    print(f"  rendezvous stale, it points at {entry.get('url')}")
elif entry:
    ok = False
    print(f"  rendezvous offline, {entry.get('reason', '')}")

try:
    health = httpx.get(f"{public_url}/health", timeout=30).json()
    print(f"  round trip ok, status {health.get('status')}, "
          f"database {'present' if health.get('database') else 'MISSING'}")
except Exception as error:
    ok = False
    print(f"  round trip failed, {type(error).__name__}")

print(f"\n  interface  {WORKER}" if ok else f"\n  the pipeline works, the public link does not")
print(f"  api docs   {public_url}/docs")

### Leave this cell running

It holds the session open. Interrupt it to stop.

In [ ]:
print("Serving. Interrupt this cell to stop.\n")
started = time.time()
try:
    while True:
        time.sleep(60)
        alive = tunnel.poll() is None
        print(f"  {(time.time() - started) / 60:5.0f} min   tunnel {'up' if alive else 'DOWN'}",
              flush=True)
        if not alive:
            print("  the tunnel stopped, re-run the cell above")
            break
except KeyboardInterrupt:
    tunnel.terminate()
    print("stopped")